In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from math import exp
import requests

df = pd.read_csv('/content/drive/MyDrive/DS teamproj/final_pm10_with_urbanforest.csv')



In [ ]:
# 시간 파생 변수 생성
df["timestamp"] = pd.to_datetime(df["ymdt"], format="%Y%m%d%H%M")
df["hour"] = df["timestamp"].dt.hour
df["month"] = df["timestamp"].dt.month
df["weekday"] = df["timestamp"].dt.weekday

# 거리감쇠 기반 가중 평균 feature 생성
def exp_decay_weight(area, distance, k=0.3):
    return area * np.exp(-k * distance)

w1 = exp_decay_weight(df["park1_area"], df["park1_distance"])
w2 = exp_decay_weight(df["park2_area"], df["park2_distance"])
w3 = exp_decay_weight(df["park3_area"], df["park3_distance"])
total_w = w1 + w2 + w3

df["weighted_area"] = (df["park1_area"] * w1 + df["park2_area"] * w2 + df["park3_area"] * w3) / total_w
df["weighted_distance"] = (df["park1_distance"] * w1 + df["park2_distance"] * w2 + df["park3_distance"] * w3) / total_w



In [ ]:
def get_weather(dt):
    import requests
    from numpy import nan
    base_date_time = dt
    url = f'https://apihub.kma.go.kr/api/typ01/url/kma_sfctm2.php?tm={base_date_time}&stn=108&help=0&authKey=YS6kErh3SmeupBK4d2pnjg'

    try:
        response = requests.get(url)
        lines = response.text.splitlines()

        data_line = next(line for line in lines if not line.startswith('#'))
        values = data_line.split()

        def safe_float(val):
            try:
                f = float(val)
                return nan if f == -9.0 else f
            except:
                return nan

        def rain_float(val):
            try:
                f = float(val)
                return 0.0 if f == -9.0 else f  # 🌧️ -9.0은 비가 안 온 것
            except:
                return nan

        return {
            'wind_dir': safe_float(values[2]),
            'wind_spd': safe_float(values[3]),
            'rain': rain_float(values[15]),
            'hPA': safe_float(values[7]),
            'temperature': safe_float(values[11]),
            'humidity': safe_float(values[13]),
        }

    except Exception as e:
        print(f"⚠️ API error for {dt}: {e}")
        return {
            'wind_dir': nan,
            'wind_spd': nan,
            'rain': nan,
            'hPA': nan,
            'temperature': nan,
            'humidity': nan
        }


In [ ]:
unique_times = df['ymdt'].drop_duplicates().tolist()
print(unique_times)

[202401010300, 202401010800, 202401011400, 202401011800, 202401030300, 202401030800, 202401031400, 202401031800, 202401050300, 202401050800, 202401051400, 202401051800, 202401060300, 202401060800, 202401061400, 202401061800, 202401080300, 202401080800, 202401081400, 202401081800, 202401100300, 202401100800, 202401101400, 202401101800, 202401120300, 202401120800, 202401121400, 202401121800, 202401130300, 202401130800, 202401131400, 202401131800, 202401150300, 202401150800, 202401151400, 202401151800, 202401170300, 202401170800, 202401171400, 202401171800, 202401190300, 202401190800, 202401191400, 202401191800, 202401200300, 202401200800, 202401201400, 202401201800, 202401220300, 202401220800, 202401221400, 202401221800, 202401240300, 202401240800, 202401241400, 202401241800, 202401260300, 202401260800, 202401261400, 202401261800, 202401270300, 202401270800, 202401271400, 202401271800, 202401290300, 202401290800, 202401291400, 202401291800, 202401310300, 202401310800, 202401311400, 20240

In [ ]:
# 각 날짜시간에 대해 API 호출해서 결과 저장
weather_dict = {}
for dt in unique_times:
    weather_dict[dt] = get_weather(dt)

# 원래 df에 날짜시간 기준으로 매핑 (broadcast)
weather_df = pd.DataFrame.from_dict(weather_dict, orient='index')
weather_df.columns = ['wind_dir', 'wind_spd', 'rain', 'hPA', 'temperature', 'humidity']
weather_df['ymdt'] = weather_df.index

# 원래 df에 병합
df = df.merge(weather_df, on='ymdt', how='left')

⚠️ API error for 202403110800: 


In [ ]:
# CSV 저장
df.to_csv('/content/drive/MyDrive/DS teamproj/train.csv', index=False)